# Task 1.1 — Pipeline char-level tiny_shakespeare

## Imports y reproducibilidad

In [1]:
import os
import random
import ssl
import urllib.request

import certifi

import numpy as np
import torch

SEED = 1337
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
if torch.backends.mps.is_available():
    torch.mps.manual_seed(SEED)

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print("device:", device)

device: mps


## Hiperparámetros

In [2]:
block_size = 128
d_model = 128
n_heads = 4
n_layers = 3
batch_size = 64
lr = 3e-4
epochs = 10

## Descarga del dataset

In [3]:
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
DATA_PATH = os.path.join("data", "input.txt")

if not os.path.exists(DATA_PATH):
    os.makedirs("data", exist_ok=True)
    ctx = ssl.create_default_context(cafile=certifi.where())
    with urllib.request.urlopen(URL, context=ctx) as r, open(DATA_PATH, "wb") as f:
        f.write(r.read())

with open(DATA_PATH, "r", encoding="utf-8") as f:
    text = f.read()

print("Número de caracteres:", len(text))
print(text[:300])

Número de caracteres: 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


## Vocabulario y mappings

In [4]:
chars = sorted(set(text))
vocab_size = len(chars)
assert vocab_size == 65, vocab_size

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):
    return [stoi[c] for c in s]

def decode(ids):
    return "".join(itos[int(i)] for i in ids)

print("vocab_size:", vocab_size)
print(repr("".join(chars)))

sample = "First Citizen:"
assert decode(encode(sample)) == sample
print(sample, "->", encode(sample))

vocab_size: 65
"\n !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"
First Citizen: -> [18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52, 10]


## Codificación del texto

In [5]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:50])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56])


## Split train / val (90% / 10%)

In [6]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print("train:", len(train_data), "| val:", len(val_data))

train: 1003854 | val: 111540


## Generador de batches

In [7]:
def get_batch(split, batch_size=batch_size, block_size=block_size):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

## Verificación del batch

In [8]:
xb, yb = get_batch("train")
print("x:", xb.shape, "| y:", yb.shape)
assert xb.shape == (batch_size, block_size)
assert yb.shape == (batch_size, block_size)
assert torch.equal(xb[:, 1:], yb[:, :-1])

xv, yv = get_batch("val")
assert xv.shape == yv.shape == (batch_size, block_size)

x0, y0 = xb[0].cpu(), yb[0].cpu()
for t in range(8):
    print(f"{decode(x0[:t + 1])!r} -> {decode([y0[t]])!r}")

x: torch.Size([64, 128]) | y: torch.Size([64, 128])


' ' -> 'i'
' i' -> 'f'
' if' -> ' '
' if ' -> 'd'
' if d' -> 'i'
' if di' -> 'e'
' if die' -> ','
' if die,' -> ' '


## Pasos por epoch

In [9]:
steps_per_epoch = len(train_data) // (batch_size * block_size)
total_steps = steps_per_epoch * epochs
print("steps_per_epoch:", steps_per_epoch, "| total_steps:", total_steps)

steps_per_epoch: 122 | total_steps: 1220
